In [2]:
import pandas as pd
import sqlite3
csgo_table = pd.read_csv("data/csgo_games.csv")
connection_sql = sqlite3.connect("data/csgo_stats.sqlite")

In [14]:
csgo_table.info

<bound method DataFrame.info of       match_date       team_1       team_2  t1_points  t2_points  \
0     2016-12-18       CLOUD9  HELLRAISERS         13         16   
1     2016-12-18    IMMORTALS           G2         17         19   
2     2016-12-18  MOUSESPORTS    IMMORTALS         16          3   
3     2016-12-18     DIGNITAS           G2         16          9   
4     2016-12-18        OPTIC  HELLRAISERS         16         10   
...          ...          ...          ...        ...        ...   
3782  2020-01-25   VIRTUS.PRO       CLOUD9          0          2   
3783  2020-01-24       HEROIC    MAD LIONS         10         16   
3784  2020-01-19       HEROIC        FORZE          0          2   
3785  2020-01-19        CR4ZY        FORZE          0          2   
3786  2020-01-10       CLOUD9         MIBR          3          1   

      t1_world_rank  t2_world_rank  t1_h2h_win_perc  t2_h2h_win_perc winner  \
0                 9             20         0.500000         0.500000    

In [ ]:
team1_cnt = csgo_table["team_1"].value_counts()
team2_cnt = csgo_table["team_2"].value_counts()
print(team1_cnt)

In [12]:
team2_cnt

team_2
NIP                      251
G2                       247
MOUSESPORTS              240
FAZE                     239
ASTRALIS                 215
                        ... 
AGG                        1
?                          1
MISFITS                    1
QUANTUM BELLATOR FIRE      1
C0NTACT                    1
Name: count, Length: 66, dtype: int64

In [32]:
csgo_table_2 = csgo_table[["team_1","team_2","t1_world_rank","t2_world_rank","winner"]]
csgo_table_2

,team_1,team_2,t1_world_rank,t2_world_rank,winner
0,CLOUD9,HELLRAISERS,9,20,t2
1,IMMORTALS,G2,13,10,t2
2,MOUSESPORTS,IMMORTALS,12,13,t1
3,DIGNITAS,G2,6,10,t1
4,OPTIC,HELLRAISERS,4,20,t1
...,...,...,...,...,...
3782,VIRTUS.PRO,CLOUD9,19,17,t2
3783,HEROIC,MAD LIONS,16,18,t2
3784,HEROIC,FORZE,16,13,t2
3785,CR4ZY,FORZE,20,13,t2


In [75]:
csgo_table_3 = csgo_table_2.copy()

# for key, row in csgo_table_3.iterrows():
#     if row["winner"] == csgo_table_3.iloc[0,4]:
#         csgo_table_3["winner_has_less_rank"] = csgo_table_3["t1_world_rank"] > csgo_table_3["t2_world_rank"]
#     else:
#         csgo_table_3["winner_has_less_rank"] = csgo_table_3["t1_world_rank"] < csgo_table_3["t2_world_rank"]

csgo_table_3["winner_has_less_rank"] = (
    (csgo_table_3["winner"] == "t1") & (csgo_table_3["t1_world_rank"] < csgo_table_3["t2_world_rank"])
) | (
    (csgo_table_3["winner"] == "t2") & (csgo_table_3["t2_world_rank"] < csgo_table_3["t1_world_rank"])
)
csgo_table_3["winner_is_t1"] = (csgo_table_3["winner"] == "t1")
csgo_table_3

,team_1,team_2,t1_world_rank,t2_world_rank,winner,winner_has_less_rank,winner_is_t1
0,CLOUD9,HELLRAISERS,9,20,t2,False,False
1,IMMORTALS,G2,13,10,t2,True,False
2,MOUSESPORTS,IMMORTALS,12,13,t1,True,True
3,DIGNITAS,G2,6,10,t1,True,True
4,OPTIC,HELLRAISERS,4,20,t1,True,True
...,...,...,...,...,...,...,...
3782,VIRTUS.PRO,CLOUD9,19,17,t2,True,False
3783,HEROIC,MAD LIONS,16,18,t2,False,False
3784,HEROIC,FORZE,16,13,t2,True,False
3785,CR4ZY,FORZE,20,13,t2,True,False


In [162]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

X = csgo_table_3[["t1_world_rank","t2_world_rank"]]  
y = csgo_table_3["winner_is_t1"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("mse:", mse)
print(model.coef_)      
print(model.intercept_) 


mse: 0.2418477095741911
[-0.01376125  0.0119507 ]
0.5036150392833214


In [163]:

test_data = pd.DataFrame({
    "t1_world_rank": [9,13,19,16,16,20,18,4,6],
    "t2_world_rank": [20,10,17,18,13,13,14,20,10],
})

for i,row in test_data.iterrows():
    prediction = model.predict(pd.DataFrame([{"t1_world_rank": row["t1_world_rank"],
                                             "t2_world_rank": row["t2_world_rank"]}]))
    
    print(True if prediction[0] > 0.506 else False)




True
False
False
False
False
False
False
True
True


In [186]:
test_data = csgo_table_3[["t1_world_rank","t2_world_rank"]]  

results = []
for i,row in test_data.iterrows():
    prediction = model.predict(pd.DataFrame([{"t1_world_rank": row["t1_world_rank"],
                                             "t2_world_rank": row["t2_world_rank"]}]))
    
    results.append(True if prediction[0] > 0.49 else False)

In [187]:
original_results = csgo_table_3["winner_is_t1"]

comparing_errors = 0
for i,value in enumerate(results):
    if value != original_results[i]:
        comparing_errors += 1

In [188]:
f"{(len(results)-comparing_errors)/len(results)*100:.3f}%"

'58.938%'